# QDAC sourced NI DAQ CrossTalk Frequency Depdendency Test

## Import Libraries

In [1]:
import time
import json
import pyvisa
import numpy as np
import matplotlib.pyplot as plt
from time import sleep

from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qcodes.instrument.channel import ChannelList

from qstl_instruments.qstl_qdac2 import QSTL_QDac2
from qstl_instruments.qstl_nidaq import QSTL_NIDaq

from pyvisa.constants import StopBits, Parity

rm = pyvisa.ResourceManager()
rm.list_resources()

('ASRL1::INSTR',
 'ASRL3::INSTR',
 'ASRL5::INSTR',
 'ASRL20::INSTR',
 'ASRL21::INSTR',
 'ASRL22::INSTR',
 'ASRL23::INSTR',
 'ASRL24::INSTR',
 'ASRL25::INSTR',
 'ASRL26::INSTR',
 'ASRL27::INSTR')

## Instantiation of Instruments

In [2]:
contacts = {
    "X" : 1,
    "Y" : 2
}

ai_chans = {
    "I1" : "Dev2/ai0",
    "I2" : "Dev2/ai2"
}

# Set up database
initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251009_QDAC_NIQAC_synctest/QDAC_IQ_Mod_Bandwidth_Test.db")

qdac2 = QSTL_QDac2(
    name = "qdac2",
    address = "ASRL5::INSTR",
    ramp_rate = 1,
    i_threshold = 2e-9,
    v_limit = 0.5,
    contacts = contacts
)
station = Station(qdac2)

daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)

qdac2.ramp_all_channels_to_zero()
qdac2.get_initial_voltages()

Connected to: QDevil QDAC-II (serial:368, firmware:13-1.57) in 0.09s


{'X': 0.0, 'Y': 0.0}

## Instruments Setup

In [8]:
## setup qdac 2 for the 2D sweep
qdac2.free_all_triggers()
qdac2.ext3.delay_s(0)
qdac2.v_limit = 1.2

device1 = "I1"
device2 = "I2"

X = ["X"]
Y = ["Y"]

freqs = np.linspace(1000, 400000, 50)
acq_time = 2000e-6
times = np.linspace(0, acq_time, int(acq_time * daq.max_sampling_rate / 2))

qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")

exp = load_or_create_experiment("2D sweep", "QDAC+NiDAQ_Xtalk_Freq_Dep_Test")
meas = Measurement(exp=exp, station=station)

I1_multi = Parameter(name= "I1_multi", label="I1_multi", unit="V")
I2_multi = Parameter(name= "I2_multi", label="I2_multi", unit="V")
I1_single = Parameter(name= "I1_single", label="I1_single", unit="V")
I2_single = Parameter(name= "I2_single", label="I2_single", unit="V")
R_single = Parameter(name = "rho_xy_single", label = "rho_xy_single", unit = "AU")
R_multi = Parameter(name = "rho_xy_multi", label = "rho_xy_multi", unit = "AU")
t = Parameter(name = "t",label="t", unit = "s" )
f = Parameter(name = "f",label="f", unit = "Hz" )

meas.register_parameter(t)
meas.register_parameter(f)
meas.register_parameter(I1_multi, setpoints = (t, f))
meas.register_parameter(I2_multi, setpoints = (t, f))
meas.register_parameter(R_multi, setpoints = (t, f))
meas.register_parameter(I1_single, setpoints = (t, f))
meas.register_parameter(I2_single, setpoints = (t, f))
meas.register_parameter(R_single, setpoints = (t, f))

## Correlation Function

In [9]:
def crosscorr_rho(x: np.array, y: np.array) -> np.array:
    x = np.asarray(x)
    y = np.asarray(y)

    N = x.size

    lags = np.linspace(0, N-1, N,  dtype=int)

    px  = np.concatenate(([0], np.cumsum(x)))
    py  = np.concatenate(([0], np.cumsum(y)))
    px2 = np.concatenate(([0], np.cumsum(np.abs(x)**2)))
    py2 = np.concatenate(([0], np.cumsum(np.abs(y)**2)))

    rho = np.empty_like(lags, dtype=np.float64)
    rxx = np.vdot(x, x)
    
    ryy = np.vdot(y, y)

    for idx, k in enumerate(lags):
        n = N - k
        x0, x1 = 0, n - 1
        y0, y1 = k, N - 1

        x_seg = x[x0:x1]
        y_seg = y[y0:y1]

        rxy = np.vdot(x[x0:x1], y[y0:y1])  # vdot does conjugate on the first argument

        rho[idx] = rxy/np.sqrt(np.abs(rxx * ryy))
    return rho


## Crosstalk Measurement

In [16]:
qdac2.free_all_triggers()
InitialConditions = qdac2.get_initial_voltages()

start_time = time.time()
with meas.run() as datasaver:
    loop_counter = 0
    datasaver.dataset.add_metadata(tag="Contacts", metadata=json.dumps(contacts))
    datasaver.dataset.add_metadata(tag="IC", metadata=json.dumps(InitialConditions))
    datasaver.dataset.add_metadata(
        tag = "Sweep_params",
        metadata = json.dumps(
            {
                "X" : X,
                "Y" : Y,
                "freqs" : list(freqs),
                "acq_time" : acq_time
            }
        )
    )
    for freq in freqs:
        qdac2.ramp_all_channels_to_zero()
        qdac2.free_all_triggers()
        sine = qdac2.ch01.sine_wave(
            period_s = 1/freq,
            span_V = 1,
            offset_V = 0.0
        )
        trig = sine.start_marker()
        qdac2.ext5.width_s(2e-6)
        qdac2.ext5.source_from_trigger(trig)
        # Read NI Daq traces
        result = daq.read_triggered_multi_channels(
            sine,
            [ai_chans[device1], ai_chans[device2]],
            int(acq_time * daq.max_sampling_rate / 2),
            -1,
            +1,
            int(acq_time * daq.max_sampling_rate / 2) + 1
        )
        result_0 = result[0]
        result_1 = result[1]

        datasaver.add_result(
            (t, np.linspace(0, acq_time, len(result_0))),
            (f, [freq] * len(result_0)),
            (R_multi, crosscorr_rho(result_1, result_0)),
            (I1_multi, result_0),
            (I2_multi, result_1),
        )

        
        qdac2.ramp_all_channels_to_zero()
        qdac2.free_all_triggers()
        sine = qdac2.ch01.sine_wave(
            period_s = 1/freq,
            span_V = 1,
            offset_V = 0.0
        )
        trig = sine.start_marker()
        qdac2.ext5.width_s(2e-6)
        qdac2.ext5.source_from_trigger(trig)

        result_0 = daq.read_triggered_voltage(
            sine,
            ai_chans[device1],
            int(acq_time * daq.max_sampling_rate),
            -1,
            +1,
            int(acq_time * daq.max_sampling_rate) + 1
        )

        qdac2.ramp_all_channels_to_zero()
        qdac2.free_all_triggers()
        sine = qdac2.ch01.sine_wave(
            period_s = 1/freq,
            span_V = 1,
            offset_V = 0.0
        )
        trig = sine.start_marker()
        qdac2.ext5.width_s(2e-6)
        qdac2.ext5.source_from_trigger(trig)
        result_1 = daq.read_triggered_voltage(
            sine,
            ai_chans[device2],
            int(acq_time * daq.max_sampling_rate),
            -1,
            +1,
            int(acq_time * daq.max_sampling_rate) + 1
        )

        datasaver.add_result(
            (t, np.linspace(0, acq_time, len(result_0) >> 1)),
            (f, [freq] * (len(result_0) >> 1)),
            (R_single, crosscorr_rho(result_1, result_0).reshape(-1, 2).mean(axis=1)),
            (I1_single, result_0.reshape(-1, 2).mean(axis=1)),
            (I2_single, result_1.reshape(-1, 2).mean(axis=1)),
        )
        loop_counter = loop_counter+1
        print(f'Time elapsed: {np.round(time.time()-start_time, 2)} sec. Loop finished: {loop_counter}/{len(freqs)}.')

end_time = time.time()
print(f'Time elapsed: {np.round(end_time-start_time, 2)} sec.')

qdac2.channels[0:23].dc_slew_rate_V_per_s(1)

qdac2.ramp_all_channels_to_zero()

Starting experimental run with id: 153. 
Time elapsed: 0.14 sec. Loop finished: 1/50.
Time elapsed: 0.24 sec. Loop finished: 2/50.
Time elapsed: 0.33 sec. Loop finished: 3/50.
Time elapsed: 0.42 sec. Loop finished: 4/50.
Time elapsed: 0.52 sec. Loop finished: 5/50.
Time elapsed: 0.61 sec. Loop finished: 6/50.
Time elapsed: 0.71 sec. Loop finished: 7/50.
Time elapsed: 0.8 sec. Loop finished: 8/50.
Time elapsed: 0.9 sec. Loop finished: 9/50.
Time elapsed: 0.99 sec. Loop finished: 10/50.
Time elapsed: 1.09 sec. Loop finished: 11/50.
Time elapsed: 1.18 sec. Loop finished: 12/50.
Time elapsed: 1.27 sec. Loop finished: 13/50.
Time elapsed: 1.37 sec. Loop finished: 14/50.
Time elapsed: 1.46 sec. Loop finished: 15/50.
Time elapsed: 1.55 sec. Loop finished: 16/50.
Time elapsed: 1.66 sec. Loop finished: 17/50.
Time elapsed: 1.76 sec. Loop finished: 18/50.
Time elapsed: 1.85 sec. Loop finished: 19/50.
Time elapsed: 1.95 sec. Loop finished: 20/50.
Time elapsed: 2.04 sec. Loop finished: 21/50.
Time